# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [ ]:
import os
import re
import json
import time

import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_colwidth', 200)

/Users/aoshavrina/Library/Application Support/Claude/scratch-workspaces/94c4cd5c-c8b3-4e58-bdf9-78fdcc68367e/979b9d52-e624-4691-b6f1-08d42a4da031/scratch-2026-10-07-362e92/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Языки: польский (`pl`), словацкий (`sk`), немецкий (`de`), нидерландский (`nl`), казахский (`kk`). Две пары близкородственных языков на латинице и один кириллический.

In [ ]:
HEADERS = {'User-Agent': 'HSE-NLP-homework/0.1 (shavrina157@gmail.com)'}


def wiki_api(lang, retries=5, **params):
    params = {'action': 'query', 'format': 'json', 'formatversion': 2, **params}
    for attempt in range(retries):
        r = requests.get(f'https://{lang}.wikipedia.org/w/api.php', params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:
            time.sleep(int(r.headers.get('Retry-After', 2 ** attempt)))
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


def get_texts_for_lang(lang, n=10):
    titles = [p['title'] for p in wiki_api(lang, list='random', rnnamespace=0, rnlimit=n)['query']['random']]
    texts = []
    for title in titles:
        # полный текст через extracts отдается только по одной статье за запрос
        page = wiki_api(lang, prop='extracts', explaintext=1, titles=title, redirects=1)['query']['pages'][0]
        texts.append({'title': title, 'text': page.get('extract', '')})
        time.sleep(0.35)
    return texts


langs = ['pl', 'sk', 'de', 'nl', 'kk']
WIKI_CACHE = 'wiki_hw.json'

if os.path.exists(WIKI_CACHE):
    wiki = json.load(open(WIKI_CACHE))
else:
    wiki = {lang: get_texts_for_lang(lang) for lang in langs}
    json.dump(wiki, open(WIKI_CACHE, 'w'), ensure_ascii=False)

{lang: [(a['title'], len(a['text'])) for a in arts] for lang, arts in wiki.items()}

{'pl': [('Jan Raczyborski', 1932),
  ('Trapt', 6568),
  ('Bucharest Financial Plaza', 292),
  ('Home Nations Championship 1905', 664),
  ('Majdan (rejon dubieński)', 517),
  ('Leszek Guzicki', 3520),
  ('Azjos z Samos', 392),
  ('Jaszowice-Kolonia', 496),
  ('Anatol Jurewicz', 1282),
  ('Heinz Fritz Köhler', 2048)],
 'sk': [('S1m0ne', 298),
  ('Siladská župa', 155),
  ('Saint-Georges-Montcocq', 502),
  ('Le Monocle', 604),
  ('Gazebo', 1513),
  ('Focke-Achgelis Fa 223', 3692),
  ('Le Luart', 479),
  ('Jelení (Nová Pec)', 760),
  ('Alexander III. (Byzantská ríša)', 971),
  ('R Crucis', 51)],
 'de': [('Malabar (Schiff, 1925)', 1615),
  ('Räuberrad', 2016),
  ('Deutscher Germanistenverband', 8289),
  ('Krems II', 1236),
  ('Patrick Weimar', 1466),
  ('NGC 5930', 1153),
  ('Ferroviário Maputo', 5044),
  ('Zweybrück', 320),
  ('René Falk Thomasius', 1778),
  ('Lost (Linkin-Park-Lied)', 2280)],
 'nl': [('Hyphydrus madagascariensis', 188),
  ('Leptaulax sambawae', 162),
  ('Feel the Love (Rud

In [ ]:
print(wiki['sk'][2]['text'])

Saint-Georges-Montcocq je francúzska obec, ktorá sa nachádza v departemente Manche, v regióne Dolná Normandia.


== Poloha ==
Obec má rozlohu 8,94 km². Najvyšší bod je položený 126 m n. m. a najnižší bod 7 m n. m.


== Obyvateľstvo ==
Počet obyvateľov obce je 824 (2011).
Nasledujúci graf zobrazuje vývoj počtu obyvateľov v obci.

Zdroje: INSEE


== Referencie ==


== Pozri aj ==
Zoznam obcí departementu Manche


== Iné projekty ==

 Commons ponúka multimediálne súbory na tému Saint-Georges-Montcocq


Что чистим:
- заголовки разделов `== ... ==`;
- служебные разделы (источники, ссылки, «см. также») целиком вместе с подразделами;
- строки, которые не заканчиваются знаком конца предложения: списки, треклисты, таблицы, коды, подписи;
- пустые скобки, которые остаются на месте шаблонов, лишние пробелы и пробелы перед пунктуацией.

На предложения делим моделью SaT из `wtpsplit`: `pysbd` резал польский по `m.in.` и порядковым числительным (`zajął 3. miejsce`).

In [ ]:
SERVICE_SECTIONS = {
    'pl': {'Przypisy', 'Bibliografia', 'Linki zewnętrzne', 'Zobacz też'},
    'sk': {'Referencie', 'Pozri aj', 'Externé odkazy', 'Iné projekty', 'Zdroj', 'Literatúra'},
    'de': {'Einzelnachweise', 'Weblinks', 'Literatur', 'Siehe auch', 'Anmerkungen'},
    'nl': {'Externe link', 'Externe links', 'Bronnen', 'Zie ook', 'Referenties', 'Bronnen, noten en/of referenties'},
    'kk': {'Дереккөздер', 'Сыртқы сілтемелер', 'Тағы қараңыз', 'Әдебиет', 'Сілтемелер'},
}
HEADING = re.compile(r'^(=+)\s*(.*?)\s*\1$')
SENT_END = re.compile(r'[.!?…]["»“”)\]]*$')


def clean_text(text, lang):
    lines, skip_level = [], None
    for line in text.split('\n'):
        line = line.strip()
        m = HEADING.match(line)
        if m:
            level = len(m.group(1))
            if skip_level is not None and level <= skip_level:
                skip_level = None
            if skip_level is None and m.group(2) in SERVICE_SECTIONS[lang]:
                skip_level = level
            continue
        if skip_level is not None or not SENT_END.search(line):
            continue
        line = re.sub(r'\(\s*[,;:]?\s*\)', '', line)
        line = re.sub(r'\s+', ' ', line)
        line = re.sub(r'\s+([.,;:!?)])', r'\1', line)
        lines.append(line)
    return '\n'.join(lines)


print(clean_text(wiki['sk'][2]['text'], 'sk'))

Saint-Georges-Montcocq je francúzska obec, ktorá sa nachádza v departemente Manche, v regióne Dolná Normandia.
Obec má rozlohu 8,94 km². Najvyšší bod je položený 126 m n. m. a najnižší bod 7 m n. m.
Počet obyvateľov obce je 824 (2011).
Nasledujúci graf zobrazuje vývoj počtu obyvateľov v obci.


In [ ]:
from wtpsplit import SaT

sat = SaT('sat-12l-sm')

rows = []
for lang, arts in wiki.items():
    for a in arts:
        for par in clean_text(a['text'], lang).split('\n'):
            for sent in sat.split(par):
                sent = sent.strip()
                if len(re.findall(r'\w+', sent)) >= 3:
                    rows.append({'lang': lang, 'title': a['title'], 'sentence': sent})

df = pd.DataFrame(rows)
print(df.lang.value_counts())
df.sample(10, random_state=0)

lang
de    144
pl    128
sk     69
kk     69
nl     50
Name: count, dtype: int64


,lang,title,sentence
329,de,Lost (Linkin-Park-Lied),"Ursprünglich wurde es während der Sessions zu ihrem zweiten Studioalbum Meteora (2003) aufgenommen und später, am 10. Februar 2023, offiziell als Leadsingle aus der Neuauflage zum 20-jährigen Jubi..."
153,sk,Focke-Achgelis Fa 223,Viacero dokončených a rozpracovaných strojov bolo zničených a výroba musela byť z pôvodnej továrne presťahovaná na bezpečnejšie miesto.
135,sk,Le Monocle,"Režisérom trilógie bol Georges Lautner, ktorý išiel podľa scenára Jacquesa Roberta."
78,pl,Leszek Guzicki,Redagował dwie gazety związane z tym ruchem: „Mosty” oraz „Młodą Demokrację”.
452,kk,Қазақстан қазақтарының фамилиялары тізімі (5001-10000),Қазақстан Республикасы Ұлттық статистика бюросының мәліметі бойынша.
71,pl,Leszek Guzicki,"Leszek Stefan Guzicki (ur. 19 stycznia 1917 w Krzywym Rogu, zm. 23 lutego 1984 w Warszawie) – polski ekonomista i działacz polityczny, wykładowca Politechniki Warszawskiej, poseł do Krajowej Rady ..."
284,de,Ferroviário Maputo,"Fußball, Leichtathletik, Basketball, Radfahren, Gymnastik, Hockey, Schwimmen und Tischtennis zählen zu den Sportarten die bei Ferroviário ausgeübt werden."
220,de,Deutscher Germanistenverband,Der Verband gliedert sich in die „Gesellschaft für Hochschulgermanistik im Deutschen Germanistenverband“ (GfH im DGV) und den „Fachverband Deutsch im Deutschen Germanistenverband“ (FV im DGV).
339,de,Lost (Linkin-Park-Lied),Es wurde von den Produzenten und Animatoren Maciej Kuciara und pplpleasr erstellt.
247,de,Deutscher Germanistenverband,Die Neugründung ging mit einer Neuausrichtung der angestrebten Ziele des Verbandes einher und sollte den Verband in seiner neuen Konstitution somit klar von den völkischen Kontexten seiner Gründun...


Кластеризуем эмбеддинги `multilingual-e5-base` (как на прошлом семинаре, с префиксом `query: `) KMeans'ом на 5 кластеров. `n_init=100`: при `n_init=10` ARI заметно скачет от сида к сиду.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

model = SentenceTransformer('intfloat/multilingual-e5-base')
X = model.encode(['query: ' + s for s in df.sentence], normalize_embeddings=True,
                 batch_size=16, show_progress_bar=False)

km = KMeans(5, n_init=100, random_state=0).fit(X)
df['cluster'] = km.labels_
print('ARI:', round(adjusted_rand_score(df.lang, df.cluster), 3))
pd.crosstab(df.lang, df.cluster)

ARI: 0.711


cluster,0,1,2,3,4
lang,,,,,
de,12,1,131,0,0
kk,0,0,0,0,69
nl,4,44,1,0,1
pl,49,0,0,78,1
sk,3,65,0,0,1


In [ ]:
df.groupby(['cluster', 'lang']).title.agg(lambda s: s.value_counts().to_dict())

cluster  lang
0        de                                                                                                                                                                              {'Lost (Linkin-Park-Lied)': 12}
         nl                                                                                                                                                                            {'Feel the Love (Rudimental)': 4}
         pl                                                                                                                                                                                                {'Trapt': 49}
         sk                                                                                                                                                                                                {'Gazebo': 3}
1        de                                                                                                           

Кластеры не совпадают с языками:
- отдельный кластер собрался по теме, а не по языку: статьи про музыку на четырех языках (Trapt, песня Linkin Park, Rudimental, Gazebo);
- польский из-за этого разделен на два кластера, а нидерландский своего кластера не получил и почти целиком попал к словацкому;
- немецкий и казахский выделились почти идеально.

Для сравнения — те же предложения, но вместо эмбеддингов символьные n-граммы:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 3)).fit_transform(df.sentence)
char_labels = KMeans(5, n_init=100, random_state=0).fit_predict(X_char)
print('ARI, char n-grams:', round(adjusted_rand_score(df.lang, char_labels), 3))

ARI, char n-grams: 0.961


### Ошибки

Ошибкой считаем предложение, язык которого не совпадает с самым частым языком в его кластере. Для трех ошибок смотрим ближайших соседей по косинусу.

In [ ]:
cluster2lang = df.groupby('cluster').lang.agg(lambda s: s.value_counts().index[0])
df['pred'] = df.cluster.map(cluster2lang)
errors = df[df.lang != df.pred]
print(f'ошибок: {len(errors)} из {len(df)}')
errors.groupby(['lang', 'pred']).size()

ошибок: 68 из 460


lang  pred
de    pl      12
      sk       1
nl    de       1
      kk       1
      pl       4
      sk      44
pl    kk       1
sk    kk       1
      pl       3
dtype: int64

In [ ]:
def neighbours(sentence, k=5):
    i = df.index[df.sentence == sentence][0]
    sims = X @ X[i]
    ranked = df.assign(sim=sims).drop(index=i).sort_values('sim', ascending=False)
    print(f'[{df.lang[i]} -> кластер {df.cluster[i]} ({df.pred[i]})] {sentence}')
    print(' ближайшие:')
    for _, r in ranked.head(k).iterrows():
        print(f'   {r.sim:.3f}  {r.lang}  {r.sentence[:110]}')
    print(' ближайшие на других языках:')
    for _, r in ranked[ranked.lang != df.lang[i]].head(3).iterrows():
        print(f'   {r.sim:.3f}  {r.lang}  {r.sentence[:110]}')
    print()


for s in ['Lost ist ein Lied der US-amerikanischen Rockband Linkin Park.',
          'Hyphydrus madagascariensis is een keversoort uit de familie waterroofkevers (Dytiscidae).',
          'W 1963 dał egzamin maturalny w Coburgu.']:
    neighbours(s)

[de -> кластер 0 (pl)] Lost ist ein Lied der US-amerikanischen Rockband Linkin Park.
 ближайшие:
   0.866  de  "Lost" wurde als Nu-Metal und Hard-Rock-Song mit elektronischen Elementen beschrieben.
   0.861  de  Es wurde festgestellt, dass "Lost" klanglich ähnlich wie "Breaking the Habit" und "Numb" ist.
   0.845  de  Der Song wurde ursprünglich 2002 für das zweite Studioalbum von Linkin Park, Meteora (2003), aufgenommen und f
   0.818  de  Mike Shinoda schrieb: „Lost“ zu finden war so, als ob man ein Lieblingsfoto gefunden hätte, von dem man verges
   0.813  de  Laut Shinoda hatten die Fans seit Jahren nach Songs gefragt, die die Stimme des verstorbenen Leadsängers Chest
 ближайшие на других языках:
   0.783  nl  Het nummer is onder andere gebruikt als soundtrack in de videogame Need for Speed: Most Wanted.
   0.774  pl  Trapt – amerykańska grupa muzyczna pochodząca z Los Gatos w Kalifornii.
   0.772  nl  Feel the Love is een nummer van de Britse groep Rudimental en John Newman.

[nl 

1. **«Lost ist ein Lied der US-amerikanischen Rockband Linkin Park.»** (de → «музыкальный» кластер, где больше всего польского). Ближайшие соседи — предложения из той же немецкой статьи, и 12 из 13 ее предложений ушли в этот кластер вместе со статьями про Trapt, Rudimental и Gazebo. Ближайшие предложения на других языках — тоже про группы и альбомы. То есть ошибка на уровне всей статьи: e5 обучали сближать тексты с одинаковым смыслом независимо от языка, поэтому тема (рок-группа, альбом, сингл) перевешивает язык. Добавляют сходства и одинаковые английские названия (Linkin Park, Meteora, UK Singles Chart), которые не несут признаков языка.
2. **«Hyphydrus madagascariensis is een keversoort uit de familie waterroofkevers (Dytiscidae).»** (nl → словацкий кластер). Нидерландский остался без своего кластера: один кластер ушел на тему, второй — на польский. Нидерландские предложения в основном шаблонные заготовки про виды жуков и бабочек из латинских названий и пары служебных слов, а в словацкой части тоже много коротких заготовок (французские коммуны, звезда, фильм). Соседи этого предложения — такие же нидерландские шаблоны, то есть в эмбеддингах язык различим, но при k=5 KMeans выгоднее разрезать польский по теме, чем отделять маленький nl от sk.
3. **«W 1963 dał egzamin maturalny w Coburgu.»** (pl → казахский кластер). Короткое биографическое предложение, где главное — год и событие. Самый близкий к нему сосед вообще казахский: «1963 жылы Ленин атындағы совхозда партия комитетінің хатшысы болды» — тот же год и та же тема (карьера человека). В казахском кластере много таких фраз из биографий, и в коротком тексте год и тема дают больший вклад в вектор, чем шесть польских слов.

Итого: для задачи «определить язык» многоязычные смысловые эмбеддинги — неподходящие признаки, они специально учатся игнорировать язык. Символьные n-граммы на тех же данных дают ARI 0.96.

## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga -
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче).
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги.
Обязательно покажите примеры, которые находятся по вашим запросам.


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [ ]:
import conllu
import spacy
from collections import Counter
from spacy.tokens import Doc
from spacy.matcher import DependencyMatcher

# !wget https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu

nlp_blank = spacy.blank('ru')


def conllu_to_doc(sentence):
    words = [t for t in sentence if isinstance(t['id'], int)]
    if not words:
        return None
    position = {t['id']: k for k, t in enumerate(words)}
    return Doc(
        nlp_blank.vocab,
        words=[t['form'] for t in words],
        spaces=[(t['misc'] or {}).get('SpaceAfter') != 'No' for t in words],
        heads=[position.get(t['head'], k) for k, t in enumerate(words)],
        deps=['ROOT' if t['head'] == 0 else t['deprel'] for t in words],
        pos=[t['upos'] for t in words],
        lemmas=[t['lemma'] for t in words],
        morphs=['|'.join(f'{k}={v}' for k, v in (t['feats'] or {}).items()) for t in words],
    )


with open('ru_syntagrus-ud-train-a.conllu') as f:
    docs = [d for d in map(conllu_to_doc, conllu.parse_incr(f)) if d is not None]
len(docs)

24516

## query 1: генитив отрицания

При отрицании прямое дополнение может стоять не в винительном, а в родительном падеже: *не имеет значения*, *не принял участия*, но *не бросать дела*. Ищем переходные глаголы с дополнением и отрицанием и смотрим на падеж дополнения. Отрицание ищем по дереву: `не` зависит от самого глагола или от глагола, которому наш глагол подчинен как `xcomp` (*не мог найти ответа*).

### код

In [ ]:
neg_matcher = DependencyMatcher(nlp_blank.vocab)
neg_matcher.add('OBJ', [[
    {'RIGHT_ID': 'verb', 'RIGHT_ATTRS': {'POS': 'VERB'}},
    {'LEFT_ID': 'verb', 'REL_OP': '>', 'RIGHT_ID': 'obj', 'RIGHT_ATTRS': {'DEP': 'obj', 'POS': 'NOUN'}},
]])
neg_matcher.add('NEG', [[
    {'RIGHT_ID': 'verb', 'RIGHT_ATTRS': {'POS': 'VERB'}},
    {'LEFT_ID': 'verb', 'REL_OP': '>', 'RIGHT_ID': 'obj', 'RIGHT_ATTRS': {'DEP': 'obj', 'POS': 'NOUN'}},
    {'LEFT_ID': 'verb', 'REL_OP': '>', 'RIGHT_ID': 'neg', 'RIGHT_ATTRS': {'LEMMA': 'не', 'DEP': 'advmod'}},
]])
neg_matcher.add('NEG_XCOMP', [[
    {'RIGHT_ID': 'head', 'RIGHT_ATTRS': {'POS': {'IN': ['VERB', 'AUX', 'ADJ']}}},
    {'LEFT_ID': 'head', 'REL_OP': '>', 'RIGHT_ID': 'neg', 'RIGHT_ATTRS': {'LEMMA': 'не', 'DEP': 'advmod'}},
    {'LEFT_ID': 'head', 'REL_OP': '>', 'RIGHT_ID': 'verb', 'RIGHT_ATTRS': {'POS': 'VERB', 'DEP': 'xcomp'}},
    {'LEFT_ID': 'verb', 'REL_OP': '>', 'RIGHT_ID': 'obj', 'RIGHT_ATTRS': {'DEP': 'obj', 'POS': 'NOUN'}},
]])

QUANTIFIERS = {'много', 'мало', 'несколько', 'сколько', 'столько', 'немного', 'больше', 'меньше'}


def feat(tok, name):
    values = tok.morph.get(name)
    return values[0] if values else None


rows = []
for doc in docs:
    pairs, negated = set(), set()
    for match_id, token_ids in neg_matcher(doc):
        label = nlp_blank.vocab.strings[match_id]
        # порядок токенов в матче совпадает с порядком RIGHT_ID в шаблоне
        verb, obj = (token_ids[0], token_ids[1]) if label != 'NEG_XCOMP' else (token_ids[2], token_ids[3])
        pairs.add((verb, obj))
        if label != 'OBJ':
            negated.add((verb, obj))
    for v, o in pairs:
        verb, obj = doc[v], doc[o]
        rows.append({
            'verb': verb.lemma_, 'obj': obj.lemma_, 'neg': (v, o) in negated,
            'case': feat(obj, 'Case'), 'animacy': feat(obj, 'Animacy'), 'verbform': feat(verb, 'VerbForm'),
            # родительный после числительных и "много/мало" не связан с отрицанием
            'quantified': any(c.dep_.startswith('nummod') or c.lemma_ in QUANTIFIERS for c in obj.children),
            'phrase': f"{'не ' if (v, o) in negated else ''}{verb.text} {obj.text}",
            'sentence': doc.text,
        })

objs = pd.DataFrame(rows)
pd.crosstab(objs.neg, objs.case, margins=True)

case,Acc,Gen,Ins,Nom,All
neg,,,,,
False,9985,886,179,2,11052
True,420,492,10,0,922
All,10405,1378,189,2,11974


Родительный есть и без отрицания: числительные и *много/мало* (*пять книг*), одушевленные (у них `Acc` и `Gen` часто совпадают по форме, и генитив отрицания с ними почти не встречается), и глаголы, которые и так управляют родительным (*ждать, хотеть, достигать*). Убираем все три источника: оставляем неодушевленные дополнения без кванторов и только глаголы, у которых без отрицания дополнение в `Acc` минимум в 90% случаев.

In [ ]:
clean = objs[(objs.animacy == 'Inan') & ~objs.quantified]
acc_share = clean[~clean.neg].groupby('verb').case.agg(lambda s: (s == 'Acc').mean())
acc_verbs = acc_share[acc_share >= 0.9].index
clean = clean[clean.verb.isin(acc_verbs) & clean.case.isin(['Acc', 'Gen'])]

print(pd.crosstab(clean.neg, clean.case, margins=True), '\n')
print(pd.crosstab(clean.neg, clean.case, normalize='index').round(3))

case    Acc  Gen   All
neg                   
False  9166   22  9188
True    347  372   719
All    9513  394  9907 

case     Acc    Gen
neg                
False  0.998  0.002
True   0.483  0.517


In [ ]:
# оставшиеся родительные без отрицания - партитив
clean[~clean.neg & (clean.case == 'Gen')].phrase.tolist()

['содержит клеток',
 'дал песен',
 'Купил хлеба',
 'дал пинка',
 'понять расы',
 'Искать ответа',
 'наладить кооперации',
 'дал денег',
 'искать аналога',
 'проходят котла',
 'займет полугода',
 'добавить горючего',
 'ищет возможности',
 'создало трудностей',
 'взяли кредитов',
 'дадут денег',
 'искать пути',
 'видеть милиции',
 'вызвало споров',
 'найти примеров',
 'строить иллюзий',
 'направил сигналов']

### примеры

In [ ]:
neg = clean[clean.neg]
print('родительный:', neg[neg.case == 'Gen'].phrase.sample(12, random_state=0).tolist())
print('винительный:', neg[neg.case == 'Acc'].phrase.sample(12, random_state=0).tolist())

родительный: ['не потеряли актуальности', 'не меняя состава', 'не искала точки', 'не объяснить связи', 'не дает возможности', 'не дает оснований', 'не имеют возможности', 'не перевесит ослабления', 'не испытываете сложностей', 'не представляет Ценности', 'не любили электричества', 'не имеющее отношения']
винительный: ['не обойти клозет', 'не учитывает особенности', 'не увеличивать энтропию', 'не объяснить смысл', 'не создавать армию', 'не лоббируем интересы', 'не ограничивать деятельность', 'не замечает инструмент', 'не сдавать позиции', 'не позволить конкуренцию', 'не оставит угрозы', 'не обновить заграждения']


In [ ]:
# доля родительного по глаголам (глаголы, которые хотя бы 8 раз встретились с отрицанием)
by_verb = neg.groupby('verb').case.agg(n='size', gen_share=lambda s: (s == 'Gen').mean())
by_verb[by_verb.n >= 8].sort_values('gen_share', ascending=False).round(2)

,n,gen_share
verb,,
иметь,71,0.99
дать,19,0.84
видеть,17,0.82
давать,19,0.74
получить,15,0.67
найти,11,0.64
терять,8,0.62
решить,9,0.22


In [ ]:
pd.crosstab(neg.verbform, neg.case, margins=True)

case,Acc,Gen,All
verbform,,,
Conv,7,28,35
Fin,133,284,417
Inf,200,47,247
Part,7,13,20
All,347,372,719


In [ ]:
for form in ['Fin', 'Inf']:
    print(form, neg[neg.verbform == form].sample(8, random_state=1)[['phrase', 'case']].values.tolist())

Fin [['не начал войну', 'Acc'], ['не проводили исследований', 'Gen'], ['не произвела впечатления', 'Gen'], ['не продаст газопровод', 'Acc'], ['не оказывает влияния', 'Gen'], ['не люблю коньяк', 'Acc'], ['не чувствовал холода', 'Gen'], ['не отменял образование', 'Acc']]
Inf [['не создавать армию', 'Acc'], ['не описать трехчлен', 'Acc'], ['не пробовать силы', 'Acc'], ['не взять болгарку', 'Acc'], ['не получить болезнь', 'Acc'], ['не совершить переход', 'Acc'], ['не нарушать покой', 'Acc'], ['не отличать пораженье', 'Acc']]


Без отрицания родительный у таких дополнений почти не встречается (0.2%: партитив вроде *купил хлеба*, *дал денег* и родительный при *искать*: *искать ответа, пути*), а с отрицанием это уже половина случаев. Сильнее всего генитив отрицания закреплен за *иметь* (*не имеет значения / отношения / возможности* — почти 100%), меньше за *найти*, *получить*, *решить*. И еще одна закономерность: при финитном глаголе и деепричастии родительный преобладает, а при инфинитиве (*не бросать дела*, *не снижать налоги*) дополнение в основном остается в винительном.

## query 2: «такие Y, как X» — извлечение гиперонимов

Классический шаблон Херста: *такие города, как Москва* дает пару «Москва — это город». В SynTagRus эта конструкция размечена так: `такой` — `det` при гиперониме, гипоним — `obl` при `такой`, у гипонима `mark` *как*, а остальные элементы перечисления — `conj` при первом. Такие пары можно использовать для пополнения тезауруса или как обучающие данные для поиска гиперонимов.

### код

In [ ]:
hearst = DependencyMatcher(nlp_blank.vocab)
hearst.add('SUCH_AS', [[
    {'RIGHT_ID': 'hyper', 'RIGHT_ATTRS': {'POS': 'NOUN'}},
    {'LEFT_ID': 'hyper', 'REL_OP': '>', 'RIGHT_ID': 'such', 'RIGHT_ATTRS': {'LEMMA': 'такой', 'DEP': 'det'}},
    {'LEFT_ID': 'such', 'REL_OP': '>', 'RIGHT_ID': 'hypo', 'RIGHT_ATTRS': {'POS': {'IN': ['NOUN', 'PROPN']}}},
    {'LEFT_ID': 'hypo', 'REL_OP': '>', 'RIGHT_ID': 'kak', 'RIGHT_ATTRS': {'LEMMA': 'как', 'DEP': 'mark'}},
]])

matches = [(doc, doc[h], doc[s], doc[x]) for doc in docs for _, (h, s, x, k) in hearst(doc)]
print(len(matches))
for doc, hyper, such, hypo in matches[:12]:
    print(f'{hyper.text:>12} <- {hypo.text:<12} | {doc.text[:120]}')

46
        пост <- Семен        | За неделю начальник управления собирался позвонить Савельеву, который занимал теперь такой же пост, как и Семен Еремееви
 рационалист <- музыке       | Экономный прокофьевский стиль напоминает язык его партитур - в своих записках он такой же рационалист, как и в музыке. 
     фирмами <- БМВ          | О том, что ждет нас впереди в этой увлекательной отрасли промышленности, рассказывают в передаче канала Discovery извест
    историки <- Карамзин     | - Такие историки, как Карамзин, способствовали тому, чтобы подобное разделение считалось правильным. 
    признаки <- группы       | Существуют и более глубокие расовые признаки, такие, как группы крови. 
        зоны <- остров       | В то же время, скорее всего, потеряют привлекательность такие офшорные зоны, как остров Джерси или остров Мэн: эти терри
    факторов <- строение     | Контрастность тканей отражает особенности "внутренних", ядерных структур вещества, и она зависит от ряда таких факторов,
  

Первые же совпадения — не гипонимия, а сравнение: *такой же пост, как и Семен Еремеевич*, *такой же рационалист, как и в музыке*. Добавляем фильтры:
- при `такой` нет *же*;
- у гипонима нет *и* (*как и*) — тоже признак сравнения;
- у гипонима нет своего предлога: *нет такой пасти, как у Шелоб* сравнивает две предложные группы.

И извлекаем не одно слово, а группу целиком (*заключение под стражу*, *остров Джерси*), плюс все однородные гипонимы.

In [ ]:
def is_comparison(such, hypo):
    return (any(c.lemma_ == 'же' for c in such.children)
            or any(c.lemma_ == 'и' and c.dep_ == 'advmod' for c in hypo.children)
            or any(c.dep_ == 'case' for c in hypo.children))


def phrase(tok, drop_deps=('conj', 'cc', 'punct', 'mark', 'case', 'acl', 'acl:relcl', 'det')):
    skip = {t.i for c in tok.children if c.dep_ in drop_deps for t in c.subtree}
    skip |= {t.i for c in tok.subtree if c.dep_ == 'acl:relcl' for t in c.subtree}
    return ''.join(t.text_with_ws for t in tok.subtree if t.i not in skip).strip()


def hyponyms(hypo):
    return [hypo] + [c for c in hypo.children if c.dep_ == 'conj' and c.pos_ in ('NOUN', 'PROPN')]


pairs = []
for doc, hyper, such, hypo in matches:
    if is_comparison(such, hypo):
        continue
    for h in hyponyms(hypo):
        pairs.append({'hypernym': hyper.lemma_, 'hypernym_phrase': phrase(hyper), 'hyponym': phrase(h), 'sentence': doc.text})

pairs = pd.DataFrame(pairs)
print(len(matches), '->', pairs.sentence.nunique(), 'предложений,', len(pairs), 'пар')

46 -> 38 предложений, 81 пар


### примеры

In [ ]:
pairs.groupby('hypernym_phrase', sort=False).hyponym.agg(list).to_frame()

,hyponym
hypernym_phrase,
известными фирмами,"[БМВ, Ямаха, Тефаль, Касио, Нокиа, Панасоник]"
историки,[Карамзин]
и более глубокие расовые признаки,[группы крови]
офшорные зоны,"[остров Джерси, остров Мэн]"
факторов,"[строение вещества, взаимодействие между молекулами, молекулярное движение (диффузия, кровоток)]"
жалоб больных,"[головная боль, головокружения, снижение зрения, глухота]"
медицинском оборудовании,"[магнитные сканеры, магнитно-резонансные томографы]"
делами,[стирка]
областей робототехники,"[искусственное зрение, навигация на открытом воздухе в неструктурированном окружении, распределенный ИИ, интеллектуальное управление]"


In [ ]:
for _, r in pairs.drop_duplicates('sentence').head(5).iterrows():
    print(f'{r.hyponym} -> {r.hypernym}\n   {r.sentence}\n')

БМВ -> фирма
   О том, что ждет нас впереди в этой увлекательной отрасли промышленности, рассказывают в передаче канала Discovery известные дизайнеры Ричард Сеймур и Дик Пауэлл, работавшие с такими известными фирмами, как БМВ, Ямаха, Тефаль, Касио, Нокиа, Панасоник… 

Карамзин -> историк
   - Такие историки, как Карамзин, способствовали тому, чтобы подобное разделение считалось правильным. 

группы крови -> признак
   Существуют и более глубокие расовые признаки, такие, как группы крови. 

остров Джерси -> зона
   В то же время, скорее всего, потеряют привлекательность такие офшорные зоны, как остров Джерси или остров Мэн: эти территории являются ассоциированными с Великобританией образованиями и потому их властям придется подчиняться европейским стандартам. 

строение вещества -> фактор
   Контрастность тканей отражает особенности "внутренних", ядерных структур вещества, и она зависит от ряда таких факторов, как строение вещества, взаимодействие между молекулами, молекулярное движение

Получаются в основном правильные пары «гипоним — гипероним»: *БМВ, Ямаха, Тефаль* → *фирмы*, *Москва* → *города*, *Карамзин* → *историки*, *магнитные сканеры* → *медицинское оборудование*, *заключение под стражу* → *мера*. Ошибки связаны с тем, что шаблон все равно допускает сравнения, не помеченные *же/и* (*такую контру, как Денисов*), и с метонимией (*такое имя, как Юрий Щекочихин* — гипероним тут не «имя», а «человек»).

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения?

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя.

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу?
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет?
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



Действительно ли возможен “one pipeline to scale them all”?
Начнем с того, что какую-то аппроксимацию точно построить можно. Пайплайн из статьи выглядит достаточно солидно и внушает доверие. Плюс, авторы заявляют именно единый пайплайн и, вроде бы, не покушаются на единообразность правил. Я более-менее согласна с тем, что многие threshold-ы мы можем автоматически подбирать под каждый язык одним и тем же кодом с понятными входными данными (LID-confidence, filtering thresholds).

Некоторые другие моменты выглядят не так однозначно. Например, к счастью или к сожалению, мы не всегда можем однозначно определить, на каком языке написан текст. Иногда, кроме статистики, нам нужен какой-то социолингвистический контекст. Например, чтобы отличить “ошибку” от диалекта. Или отличить “ошибку” от код-свитчинга. А есть вообще языки без хорошо стандартизированной письменности. Вообще непонятно, как этот пайплайн будет работать, например, с шугнанским, у которого половина текстов написана на кириллице, а другая на латинице. В статье на самом деле есть даже пример такой ситуации. Для языка Дагбани из-за неверной изначальной выборки сломался и LID, и подбор стоп-слов. Они эту ошибку вычистили sanity check-ом, но для тысяч языков сделать sanity check мы не можем.

Теперь поговорим про language-agnostic фильтры. Некоторые из них на самом деле как будто базируются не на реальной характеристике языка. Например, MinHash deduplication использует word 5-граммы. С европейскими языками понятно, а что такое word n-gram для китайского языка? Тут мы упираемся в страшный вопрос, что такое слово, и результат будет сильно зависеть от выбранного алгоритма сегментации. Выбрав разные методы, мы получим разную схожесть для двух текстов. И потенциально разный вердикт, являются ли тексты дублями или нет.

Ну и очевидно, что некоторые решения придется принимать не по статистическим threshold-ам, а опираясь на наши требования к модели. Например, для моего языка больше половины доступного Web является Библией. Если я ее выкину, то у меня будет мало данных. Если я ее не выкину, то мои данные будут очень однообразны. Какую из двух зол тут выбрать? Решение надо будет принимать вне общего пайплайна.

В завершение хочу сказать, что статью не очень хочется ругать, потому что, честно говоря, я не могу придумать, как сделать этот пайплайн лучше в плане названных мной проблем. Наверное, убедиться, что наши language-agnostic фильтры действительно  language-agnostic. И попытаться консенсусно принять единообразное решение по поводу сложных случаев определения языка текста и случаев однообразных данных из одного источника. One pipeline возможен как единый framework, но не как полностью автоматическая система одинаковых решений: часть параметров можно адаптировать статистически, а некоторые решения требуют языкового контекста или вообще зависят от того, какую модель мы хотим получить.
